In [ ]:
# --- SPACEMAP notebook configuration ------------------------------------
# Every path below is built from DATA_DIR. Point it at your own data by
# setting SPACEMAP_DATA_DIR in your environment, or by editing the default.
import os
from pathlib import Path

DATA_DIR = os.environ.get("SPACEMAP_DATA_DIR", str(Path.home() / "spacemap-data"))
Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
print(f"Using DATA_DIR = {DATA_DIR}")


# Step 6B: SOM Clustering Phenotyping

This notebook performs cell phenotyping using Self-Organizing Maps (SOM) clustering on batch-corrected marker data. It provides an alternative clustering approach to Leiden with identical structure, annotations, and outputs.

## Overview

The SOM clustering approach:
1. **Data Preparation** - Load batch-corrected marker data
2. **Normalization** - Shared normalization cache with Leiden for consistency
3. **SOM Training** - Geometric sketching, self-organizing map creation, node clustering
4. **Cluster Characterization** - Generate expression heatmaps and statistics
5. **Manual annotation** - Interactive tools for phenotype assignment
6. **Quality Control** - Comprehensive validation and outputs

## Input
- Batch-corrected CSV files from Step 5 (uses *_corr markers)
- Optional: phenotype mapping for automated assignment

## Output
- CSV files with SOM cluster assignments
- CSV files with phenotype annotations
- Expression heatmaps for manual annotation
- Quality control visualizations

## Key Features
- **Intelligent clustering** - Geometric sketching for large datasets
- **Shared normalization** - Compatible cache with Leiden method
- **Comprehensive outputs** - Multiple visualization methods for validation
- **Interactive annotation** - Professional GUI for phenotype assignment

In [ ]:
# Imports & Environment Setup
import os, sys, warnings, logging
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from minisom import MiniSom
warnings.filterwarnings('ignore')

# Add src to path for imports
project_root = Path().resolve().parent
src_path = project_root / "src"
sys.path.insert(0, str(src_path))

from spacemap.config import default_config
from spacemap.ph_som_utils import (
    train_som_advanced,
    characterize_som_clusters,
    meta_cluster_clusters,
    create_meta_cluster_heatmap,
    launch_som_phenotype_assignment_app,
    apply_som_phenotypes
)

try:
    print(f'MiniSom available for SOM clustering')
except Exception:
    print('MiniSom not available; install minisom to run SOM.')

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger('step6b_som')

print(f"Project root: {project_root}")
print("Step 6B: SOM Phenotyping Setup Complete")

## Configuration

Configure the SOM clustering parameters below. These settings control the SOM grid size, training parameters, and visualization options.

In [ ]:
# =============================================================================
# CONFIGURATION - ADAPTIVE SOM (speed/quality modes)
# =============================================================================

# I/O paths
INPUT_DIR = project_root / 'outputs/batch_corrected'
OUTPUT_DIR = default_config.output_dir / 'phenotyping'
PLOTS_DIR = OUTPUT_DIR / 'som/plots'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
#input_file = INPUT_DIR / 'all_samples_combined_corrected.csv'

input_file = Path(f'{DATA_DIR}/example_study/registration/roi/cellpose/annotated_csv/corrected/batch_corrected_merged.csv')


# Normalization flags (match Leiden)
NORMALIZATION_FLAGS = dict(z1=True, z2=False, cdf=False, nlog=False)
NORMALIZATION_GROUP_COLS = ['batch_id']

# Adaptive grid sizing parameters
AUTO_SOM_GRID = True                   # If True, overrides SOM_X_DIM/SOM_Y_DIM below
TARGET_AVG_CELLS_PER_NODE = 1800       # Target training cells per SOM node (after subsample)
MIN_GRID = 40                          # Minimum side length
MAX_GRID = 70                          # Maximum side length
ASPECT_RATIO = 1.0                     # Keep square grid for now

# Speed / quality modes (choose one)
MODE = 'balanced'  # options: 'fast', 'balanced', 'high_quality'
MODE_PARAMS = {
    'fast': dict(iter_factor=0.6, start_sigma=1.3, lr=0.08, batch_size=4096),
    'balanced': dict(iter_factor=1.0, start_sigma=1.5, lr=0.10, batch_size=4096),
    'high_quality': dict(iter_factor=1.5, start_sigma=1.8, lr=0.12, batch_size=3072)
}

# Base iterations (scaled by mode & grid size)
BASE_EPOCHS = 80

# Filtering Configuration - Flexible filtering system for cell subset selection
FILTER_CRITERIA = {
    # 'tissue': 1,          # Filter to cells where tissue == 1 (use None to disable)
    'corrected': 1,       # Example: filter to corrected cells (set to None to disable)
    # 'batch_id': 'P11',   # Example: filter to specific batch (commented out)
    # 'other_column': 0,   # Example: add more filters as needed
}
LOW_EXPRESSION_Z_THRESHOLD = -0.75      # Filter out cells with globally low mean marker z-score

# Column Configuration
CELL_ID_COLUMN = 'cell_id'  # Name of the cell ID column (configurable)

# Marker sets
CORR_MARKERS = [    
    "CD31", "E-cadherin", "CD20", "CD4", "Pan-Cytokeratin", "CD68", "Podoplanin", "CD11c", "CD8", "HLA-DR",
    "CD3e", "CD45", "CD15", "FoxP3", "PDGFRa", "aSMA", "CD163", "FN1", "S100A9", "CD11b", "CD38",
    "Vimentin"
],

NON_CORR_MARKERS = [

]

MARKERS_BASE = CORR_MARKERS + NON_CORR_MARKERS
CLUSTERING_MARKERS_CORR = [f"{m}_corr" for m in CORR_MARKERS] + NON_CORR_MARKERS

# Placeholder default grid (overridden if AUTO_SOM_GRID)
SOM_X_DIM = 60
SOM_Y_DIM = 60

# Training scale parameters (will be recomputed)
SOM_ITERATIONS = BASE_EPOCHS
SOM_SIGMA = 1.5
SOM_LR = 0.10

# Mode application
if MODE not in MODE_PARAMS:
    raise ValueError(f"Unknown MODE={MODE}")
mode_cfg = MODE_PARAMS[MODE]

# Subsampling & performance
# NOTE: 'random' subsampling yields lower QE (easier measurement) and faster performance.
#       'geosketch' yields better rare cell preservation but higher QE (harder measurement).
#       Reverting to 'random' to match original notebook performance.
SUBSAMPLE_SIZE = 100000
SUBSAMPLE_METHOD = 'random' # options: 'random', 'geosketch'
MAX_PLOT_CELLS = 100000
GENERATE_PLOTS = True
RANDOM_SEED = 42

# Early stopping controls (quality vs time)
# If enabled, stop when normalized QE improves < threshold after a minimum epoch fraction.
EARLY_STOP = True
EARLY_STOP_MIN_IMPROVEMENT = 0.01
EARLY_STOP_MIN_EPOCH_FRACTION = 0.3

# Meta clustering targets (will adapt later)
SOM_N_CLUSTERS_TARGET = 50
META_CLUSTERING_METHOD = 'average'
META_CLUSTERING_METRIC = 'correlation'

# QE scaling controls
SECOND_STAGE_SCALING = False
QE_SCALING_STRATEGY = 'none'

# Adaptive grid sizing logic (training subset size unknown yet—computed later). We store parameters for later evaluation.
ADAPTIVE_CONFIG = dict(
    auto=AUTO_SOM_GRID,
    target_cells_per_node=TARGET_AVG_CELLS_PER_NODE,
    min_grid=MIN_GRID,
    max_grid=MAX_GRID,
    aspect=ASPECT_RATIO,
    mode=MODE,
    mode_params=mode_cfg,
    base_epochs=BASE_EPOCHS
)

print("Configuration loaded (adaptive):")
print(f" MODE={MODE} -> {mode_cfg}")
print(f" AUTO_SOM_GRID={AUTO_SOM_GRID} target_cells_per_node={TARGET_AVG_CELLS_PER_NODE}")
print(f" QE scaling: {QE_SCALING_STRATEGY} (second_stage={SECOND_STAGE_SCALING})")
print(f" Cell ID column: {CELL_ID_COLUMN}")
print(f" Filter criteria: {FILTER_CRITERIA}")
print(f" Low-expression threshold: mean marker z-score > {LOW_EXPRESSION_Z_THRESHOLD}")


## Step 1: Data Loading and Preparation

Load the batch-corrected data and prepare it for SOM clustering analysis.

**Shared normalization cache with Leiden method**:
- Uses identical normalization parameters for consistency across methods
- Preserves normalized data using configurable cell ID column anchoring
- Cache is shared between SOM and Leiden methods to ensure reproducible results
- Flexible filtering system via `FILTER_CRITERIA` dictionary
- Results in clean, consistent dataset ready for phenotyping

In [ ]:
# =============================================================================
# STEP 1: DATA LOADING & NORMALIZATION (SHARED CACHE WITH LEIDEN)
# =============================================================================
print("Step 1: Loading data & preparing SOM input...")
start_time = time.time()

if not input_file.exists():
    raise FileNotFoundError(f"Combined corrected data not found: {input_file}")

data = pd.read_csv(input_file, low_memory=False)
original_count = len(data)
print(f"Loaded: {original_count:,} rows")

# Apply flexible filtering criteria
pre_filter_count = len(data)
for column, value in FILTER_CRITERIA.items():
    if value is not None:  # Only apply filter if value is not None
        if column in data.columns:
            before = len(data)
            mask = data[column] == value
            data = data[mask].copy()
            filtered_count = len(data)
            print(f" {column} filter applied ({column}=={value}): {filtered_count:,}/{before:,} rows retained")
        else:
            print(f"Warning: '{column}' column not found - skipping filter")

# Verify the configurable cell ID column exists for proper anchoring
if CELL_ID_COLUMN not in data.columns:
    raise ValueError(f"Critical: '{CELL_ID_COLUMN}' column missing from input data")

print(f"Cell ID validation: {data[CELL_ID_COLUMN].nunique():,} unique cell_ids out of {len(data):,} rows")

# Keep original dataframe intact to preserve spatial coordinates
original_df = data.copy()

available_corr = [c for c in CLUSTERING_MARKERS_CORR if c in data.columns]
if not available_corr:
    raise ValueError('No corrected clustering markers found in data')

print(f"Available markers: {len(available_corr)}")

# Use shared normalization cache with Leiden method
from spacemap.ph_leiden_utils import load_or_compute_normalized

CACHE_DIR = OUTPUT_DIR / 'normalized_cache'
cache_csv = CACHE_DIR / 'normalized_markers_corrected.csv'

print(f"Using shared normalization cache: {cache_csv}")
print(f"Cache exists: {cache_csv.exists()}")

# Load or compute normalized data with proper cell ID anchoring
norm_df = load_or_compute_normalized(
    df=data[[CELL_ID_COLUMN] + NORMALIZATION_GROUP_COLS + available_corr],
    marker_cols=available_corr,
    group_cols=NORMALIZATION_GROUP_COLS,
    suffix='_norm',
    normalization_flags=NORMALIZATION_FLAGS,
    cache_path=cache_csv,
    id_col=CELL_ID_COLUMN
)

norm_cols = [c for c in norm_df.columns if c.endswith('_norm')]
print(f'Normalized columns created: {len(norm_cols)}')

# Validation: Verify cache integrity and cell ID anchoring
if CELL_ID_COLUMN in norm_df.columns:
    print(f"Cell ID anchoring verified: {norm_df[CELL_ID_COLUMN].nunique():,} unique cell_ids")
    
    # Verify cell_id alignment
    data_ids = set(data[CELL_ID_COLUMN])
    norm_ids = set(norm_df[CELL_ID_COLUMN])
    if data_ids == norm_ids:
        print("Perfect cell ID alignment between data and normalized cache")
    else:
        raise ValueError("Cell ID mismatch between data and normalized cache")
else:
    raise ValueError(f"Critical: '{CELL_ID_COLUMN}' column missing from normalized data")

# Merge normalized data back using cell ID as key
data = pd.merge(data, norm_df, on=CELL_ID_COLUMN, how='left')
original_df = pd.merge(original_df, norm_df, on=CELL_ID_COLUMN, how='left')

# Verify merge success
missing_norm = data[norm_cols].isnull().any(axis=1).sum()
if missing_norm > 0:
    print(f"Warning: {missing_norm} rows have missing normalized values")
else:
    print("All cells have complete normalized marker values")

# Apply low-expression filtering using the normalized marker panel
pre_expr_filter_count = len(data)
low_expression_scores = data[norm_cols].fillna(0).mean(axis=1)
keep_mask = low_expression_scores > LOW_EXPRESSION_Z_THRESHOLD
data = data.loc[keep_mask].copy()
original_df = original_df.loc[original_df[CELL_ID_COLUMN].isin(data[CELL_ID_COLUMN])].copy()
filtered_count = pre_expr_filter_count - len(data)
print(f"Mean marker z-score filter > {LOW_EXPRESSION_Z_THRESHOLD}: {pre_expr_filter_count:,} -> {len(data):,} cells")

# Prepare clustering data using normalized markers
clustering_data = data[norm_cols].fillna(0).astype(float)

# Scale the data for clustering
scaler = StandardScaler()
clustering_data_scaled = scaler.fit_transform(clustering_data)

# Summary
available_markers = norm_cols
total_time = time.time() - start_time
print(f"\nStep 1 Summary:")
print(f" Original rows: {original_count:,}")
print(f" After filtering: {len(data):,}")
print(f" Markers for clustering: {len(available_markers)}")
print(f" Scaled data shape: {clustering_data_scaled.shape}")
print(f" Processing time: {total_time:.1f}s")
print(f" Ready for SOM training with proper '{CELL_ID_COLUMN}' anchoring")

## Step 2: SOM Training

Train the Self-Organizing Map on the batch-corrected marker data using geometric sketching for representative subsampling.

**CRITICAL FIX: Cell_ID Anchoring**
- **Previous Issue**: SOM clusters were assigned positionally, but pd.merge operations could reorder rows, causing misalignment
- **Solution**: Modified train_som() to return cluster assignments with cell_id anchoring
- **Result**: Cluster assignments now use cell_id-based merging for 100% accurate cell-cluster mapping

### Geometric Sketching for Representative Subsampling

This SOM implementation uses **geometric sketching** for representative subsampling, identical to the Leiden method approach. This ensures:

- **Biological representativeness**: Subsample maintains diversity of the original dataset
- **Density-aware sampling**: Preserves rare and common cell populations proportionally  
- **Consistency with Leiden**: Same subsampling strategy across clustering methods
- **Better SOM training**: Representatives capture the full biological space

**Benefits over random sampling:**
- Preserves biological diversity across all cell types
- Maintains rare cell populations in training set
- Results in more biologically valid SOM nodes
- Consistent with other phenotyping methods

## Topographic Error Optimization Strategy

### **Problem Analysis**
Your high TE (~0.9939) indicates that for 99.4% of cells, their 2nd-best matching unit is NOT a neighbor of their best match on the SOM grid. This breaks the fundamental SOM principle of neighborhood preservation.

### **Root Causes Identified**
1. **Insufficient neighborhood size**: Small sigma doesn't allow proper topology formation
2. **Inadequate initialization**: Random weights don't preserve input space topology
3. **Suboptimal training schedule**: Single-phase training doesn't establish proper ordering
4. **Grid resolution mismatch**: Grid too small for data complexity

### **Comprehensive Solutions Implemented**

#### **1. PCA-Based Initialization**
- **Problem**: Random initialization ignores input space structure
- **Solution**: Initialize SOM weights along principal component directions
- **Benefit**: Starting configuration preserves major data topology

#### **2. Two-Phase Training Schedule**
- **Phase 1 (30% iterations)**: Rough ordering with large neighborhood
  - Large sigma (grid_size/2) for global topology establishment
  - High learning rate (0.5) for rapid convergence to data structure
- **Phase 2 (70% iterations)**: Fine-tuning with configured parameters
  - Target sigma for detailed neighborhood preservation
  - Lower learning rate for stability

#### **3. Optimized Parameters**
- **Sigma: 2.5 → 4.0**: Larger neighborhood for better topology preservation
- **Iterations: 1000 → 1500**: Extended training for proper convergence
- **Learning rate: 0.01 → 0.05**: Better balance between learning and stability
- **Grid: 50×50**: Sufficient resolution for data complexity

#### **4. Advanced Training Features**
- **Gaussian neighborhood**: Better than bubble function for smooth topology
- **Asymptotic decay**: More gradual parameter reduction than linear
- **TE-based early stopping**: Prevents overfitting while optimizing topology

### **Expected Improvements**
- **Target TE**: From ~0.99 to <0.5 (excellent: <0.2)
- **Better clustering**: Cells with similar biology will be spatially close
- **Reduced over-aggregation**: Improved intermediate clustering before meta-clustering
- **Publication quality**: Professional topology preservation metrics

In [ ]:
# =============================================================================
# STEP 2: ADAPTIVE SOM TRAINING (STREAMLINED)
# =============================================================================
print("Step 2: Adaptive SOM training with advanced parameters...")
start_time = time.time()

# Train SOM using the advanced function from spacemap.ph_som_utils
som_result = train_som_advanced(
    clustering_data_scaled=clustering_data_scaled,
    data_df=data,  # Original DataFrame with cell ID column
    cell_id_column=CELL_ID_COLUMN,
    adaptive_config=ADAPTIVE_CONFIG,
    mode_params=MODE_PARAMS[MODE],
    som_x_dim=SOM_X_DIM,
    som_y_dim=SOM_Y_DIM,
    subsample_size=SUBSAMPLE_SIZE,
    subsample_method=SUBSAMPLE_METHOD, # Explicitly pass method (random/geosketch)
    random_seed=RANDOM_SEED,
    eval_size=min(4000, len(clustering_data_scaled)),
    second_stage_scaling=SECOND_STAGE_SCALING,
    qe_scaling_strategy=QE_SCALING_STRATEGY,
    early_stop=EARLY_STOP,
    early_stop_min_improvement=EARLY_STOP_MIN_IMPROVEMENT,
    early_stop_min_epoch_fraction=EARLY_STOP_MIN_EPOCH_FRACTION
)

# Extract results for downstream use
som = som_result['som']
som_clusters = som_result['som_clusters']
som_clusters_df = som_result['som_clusters_df']
quantization_errors = som_result['quantization_errors']
quantization_errors_normalized = som_result['quantization_errors_normalized']
quantization_errors_kmeans_ratio = som_result['quantization_errors_kmeans_ratio']
topographic_errors = som_result['topographic_errors']
som_node_info = som_result['som_node_info']

# Update grid dimensions from training results
SOM_X_DIM, SOM_Y_DIM = som_result['grid_size']

# Merge SOM clusters with original dataframe using cell_id anchoring
if 'som_cluster' in original_df.columns:
    original_df = original_df.drop('som_cluster', axis=1)
original_df = pd.merge(original_df, som_clusters_df, on=CELL_ID_COLUMN, how='left')

# Display training summary
training_summary = som_result['training_summary']
elapsed = time.time() - start_time

print(f"\nStep 2 Summary (Advanced SOM Training):")
print(f" Grid: {training_summary['grid_size'][0]}x{training_summary['grid_size'][1]} nodes={training_summary['grid_size'][0]*training_summary['grid_size'][1]}")
print(f" Training cells: {training_summary['n_training']:,}  Eval: {training_summary['eval_size']}")
print(f" Final QE: {training_summary['final_qe']:.3f}  nQE: {training_summary['final_nqe']:.3f}  Ratio: {training_summary['final_ratio']:.2f}  TE: {training_summary['final_te']:.3f}")
print(f" Node occupancy: {som_result['node_occupancy']}")
print(f" Training mode: {training_summary['mode']}  Epochs completed: {training_summary['epochs_completed']}")
print(f" Total runtime: {elapsed/60:.2f} min")

# Quality assessment
final_nqe = training_summary['final_nqe']
final_te = training_summary['final_te']

if final_nqe < 0.20:
    print(" Quantization: Excellent")
elif final_nqe < 0.35:
    print(" Quantization: Good") 
elif final_nqe < 0.55:
    print(" Quantization: Moderate")
else:
    print(" Quantization: Needs improvement")

if final_te < 0.2:
    print(" Topology: Excellent")
elif final_te < 0.5:
    print(" Topology: Acceptable") 
else:
    print(" Topology: Weak")

print(f"\nSOM training complete with proper '{CELL_ID_COLUMN}' anchoring")

In [ ]:
# =============================================================================
# VALIDATION: Cell_ID Anchoring Verification
# =============================================================================
print("Validating cell_id anchoring implementation...")


# Ensure som_result has all required keys for compatibility with visualization code
som_result['n_som_clusters'] = len(som_node_info)
som_result['checkpoints'] = som_result.get('checkpoints', list(range(len(quantization_errors))))

print(f"SOM result validated with n_som_clusters={som_result['n_som_clusters']}")


# Check 1: Verify that som_clusters_df uses cell_id anchoring
print(f"Cell_ID anchoring verification:")
print(f"  som_clusters_df shape: {som_clusters_df.shape}")
print(f"  som_clusters_df columns: {list(som_clusters_df.columns)}")
print(f"  Cell_ids in som_clusters_df: {som_clusters_df[CELL_ID_COLUMN].nunique():,} unique")

# Check 2: Verify merge success
print(f"Merge verification:")
print(f"  original_df before merge had cell_ids: {original_df[CELL_ID_COLUMN].nunique():,} unique")
print(f"  All cells have som_cluster assignments: {original_df['som_cluster'].notna().all()}")

# Check 3: Sample verification - show that specific cell_ids got correct cluster assignments
sample_cells = som_clusters_df.head(3)
print(f"Sample cell assignments:")
for _, row in sample_cells.iterrows():
    cell_id = row[CELL_ID_COLUMN]
    som_cluster = row['som_cluster']
    # Verify this cell_id has the same cluster in original_df
    original_cluster = original_df[original_df[CELL_ID_COLUMN] == cell_id]['som_cluster'].iloc[0]
    match_status = "Pass" if som_cluster == original_cluster else "Fail"
    print(f"  {match_status} Cell {cell_id}: cluster {som_cluster} (matches: {som_cluster == original_cluster})")

print(f"\nSolution Summary:")
print(f"   Before: Clusters assigned positionally (could mismatch after pd.merge row reordering)")
print(f"   After:  Clusters assigned via cell_id anchoring (guaranteed correct alignment)")
print(f"   Result: {original_df[CELL_ID_COLUMN].nunique():,} cells correctly mapped to SOM clusters")

## Step 3: Cluster Characterization

Characterize SOM clusters and perform hierarchical meta-clustering for easier annotation.

In [ ]:
# =============================================================================
# STEP 3: CLUSTER CHARACTERIZATION
# =============================================================================
print("Step 3: Characterizing SOM clusters...")
start_time = time.time()

# Verify data integrity before characterization
print(f"Characterization input verification:")
print(f" original_df shape: {original_df.shape}")
print(f" '{CELL_ID_COLUMN}' column present: {CELL_ID_COLUMN in original_df.columns}")
print(f" 'som_cluster' column present: {'som_cluster' in original_df.columns}")
print(f" Normalized marker columns: {len([c for c in original_df.columns if c.endswith('_norm')])}")

# Characterize clusters using normalized markers
cluster_char_df = characterize_som_clusters(original_df, som_clusters, norm_cols)

# Validation
expected_cols = ['som_cluster', 'cell_count'] + [f"{col}_mean" for col in norm_cols]
missing_cols = [col for col in expected_cols if col not in cluster_char_df.columns]
if missing_cols:
    print(f"Warning: Missing columns in characterization: {missing_cols}")
else:
    print("Cluster characterization has all expected columns")

# Verify all clusters are represented
original_clusters = set(original_df['som_cluster'].unique())
char_clusters = set(cluster_char_df['som_cluster'].unique())
if original_clusters == char_clusters:
    print(f"All {len(original_clusters)} clusters properly characterized")
else:
    missing_in_char = original_clusters - char_clusters
    extra_in_char = char_clusters - original_clusters
    if missing_in_char:
        print(f"Warning: Clusters missing from characterization: {missing_in_char}")
    if extra_in_char:
        print(f"Warning: Extra clusters in characterization: {extra_in_char}")

# Save characterization
char_path = OUTPUT_DIR / "som_cluster_characterization.csv"
cluster_char_df.to_csv(char_path, index=False)

char_time = time.time() - start_time
print(f"\nStep 3 Summary:")
print(f" Clusters characterized: {len(cluster_char_df)}")
print(f" Mean cells/cluster: {cluster_char_df['cell_count'].mean():.1f}")
print(f" Characterization saved: {char_path}")
print(f" Processing time: {char_time:.1f}s")

### Step 3.1: Hierarchical Meta-Clustering

Reduce the number of SOM clusters by grouping similar ones together using hierarchical clustering for easier annotation.

In [ ]:
# =============================================================================
# STEP 3.1: META-CLUSTERING
# =============================================================================
print("Step 3.1: Meta-clustering SOM clusters...")
start_time = time.time()

meta_char_df, meta_cluster_map = meta_cluster_clusters(
    cluster_char_df.copy(),
    target_n=SOM_N_CLUSTERS_TARGET,
    method=META_CLUSTERING_METHOD,
    metric=META_CLUSTERING_METRIC
)

# Apply meta-cluster assignments to original dataframe
original_df['meta_cluster'] = original_df['som_cluster'].map(meta_cluster_map)

# Prepare char_df for downstream use
if 'meta_cluster' in meta_char_df.columns and len(meta_char_df) > 0:
    # Use meta clusters as primary clustering
    char_df = meta_char_df.copy()
    # Clean up columns for consistent interface
    char_df = char_df.drop(['som_cluster'], axis=1, errors='ignore')
    char_df = char_df.rename(columns={'meta_cluster': 'som_cluster'})
    char_df = char_df.drop(['som_clusters'], axis=1, errors='ignore')
else:
    # No meta-clustering occurred, use original characterization
    char_df = cluster_char_df.copy()

# Create expression heatmap
reordered_cluster_ids, heatmap_file = create_meta_cluster_heatmap(
    char_df,
    plots_dir=PLOTS_DIR,
    title="Meta-Cluster Expression Heatmap"
)

meta_time = time.time() - start_time
print(f"\nStep 3.1 Summary:")
print(f" Meta-clusters created: {len(meta_char_df)}")
print(f" Mean cells/meta-cluster: {meta_char_df['cell_count'].mean():.1f}")
print(f" Heatmap saved: {heatmap_file}")
print(f" Processing time: {meta_time:.1f}s")

### Step 3.2: SOM Training Quality Assessment

Comprehensive visualization of SOM training results and quality metrics.

## Understanding SOM Training Metrics

### **Quantization Error (QE)**
- **What it measures**: Average distance between input vectors and their best matching unit (BMU) in the SOM
- **Interpretation**: How well the SOM represents the original data
- **Good values**: Lower is better; typical range 0.001-0.01 for normalized data
- **Your results**: 0.0009 → 0.0005 shows excellent data representation
- **Small changes**: Normal after initial convergence; indicates stable training

### **Topographic Error (TE)**  
- **What it measures**: Fraction of input vectors for which the first and second BMUs are not adjacent on the SOM grid
- **Interpretation**: How well the SOM preserves neighborhood relationships from input space
- **Good values**: Lower is better; ideally < 0.1, acceptable < 0.5
- **Your results**: ~0.9939 indicates poor topology preservation
- **Impact**: High TE can lead to disconnected cell populations being grouped together

### **Clinical Interpretation**
- **QE**: Your excellent QE means the SOM accurately captures marker expression patterns
- **TE**: High TE suggests the current grid size may be insufficient for your data complexity
- **Solution**: The increased grid size (50x50) and improved sigma will reduce TE in future runs

In [ ]:
# =============================================================================
# STEP 3.2: SOM TRAINING QUALITY ASSESSMENT (ENHANCED)
# =============================================================================
print("Step 3.2: Generating enhanced SOM quality assessment (raw QE, normalized QE, baseline ratios)...")

# Backward compatibility guards
kmeans_baseline = None
if 'som_result' in globals() and isinstance(som_result, dict):
    kmeans_baseline = som_result.get('kmeans_qe_baseline', None)
    nqe_series = som_result.get('quantization_errors_normalized', [])
    k_ratio_series = som_result.get('quantization_errors_kmeans_ratio', [])
    checkpoints_labels = som_result.get('checkpoints', list(range(len(quantization_errors))))
else:
    nqe_series = []
    k_ratio_series = []
    checkpoints_labels = list(range(len(quantization_errors)))

# Provide defaults if missing
if not nqe_series:
    # Derive approximation if not stored
    approx_pair_mean = som_result.get('approx_pairwise_mean', None) if 'som_result' in globals() else None
    if approx_pair_mean:
        nqe_series = [qe / approx_pair_mean for qe in quantization_errors]
    else:
        nqe_series = [np.nan] * len(quantization_errors)

# Figure layout
import matplotlib.pyplot as plt
import numpy as np
fig = plt.figure(figsize=(22, 12))
plt.style.use('default')

# GridSpec
from matplotlib.gridspec import GridSpec
gs = GridSpec(2, 3, figure=fig, height_ratios=[1, 1], width_ratios=[1, 1, 1],
              hspace=0.42, wspace=0.28, top=0.90, bottom=0.08, left=0.05, right=0.96)

# Helper indices for x-axis
x_raw = list(range(len(quantization_errors)))

# Plot 1: Raw Quantization Error
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(x_raw, quantization_errors, marker='o', color='#1f77b4', linewidth=2.0, markersize=5)
ax1.set_title('Raw Quantization Error (Eval Subset)', fontsize=12, fontweight='bold', pad=16)
ax1.set_xlabel('Checkpoint', fontsize=11)
ax1.set_ylabel('QE (Euclidean Distance)', fontsize=11)
ax1.grid(True, alpha=0.3, linestyle='--')
ax1.tick_params(labelsize=9)
if quantization_errors:
    ax1.text(0.98, 0.95, f'Final QE: {quantization_errors[-1]:.3f}', transform=ax1.transAxes,
             fontsize=10, fontweight='bold', ha='right', va='top',
             bbox=dict(boxstyle='round,pad=0.35', facecolor='white', alpha=0.85))

# Plot 2: Normalized QE
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(x_raw, nqe_series, marker='s', color='#ff7f0e', linewidth=2.0, markersize=5)
ax2.set_title('Normalized QE (QE / mean pairwise distance)', fontsize=12, fontweight='bold', pad=16)
ax2.set_xlabel('Checkpoint', fontsize=11)
ax2.set_ylabel('Normalized QE', fontsize=11)
ax2.grid(True, alpha=0.3, linestyle='--')
ax2.tick_params(labelsize=9)
# Zones
ax2.axhspan(0, 0.20, color='green', alpha=0.12, label='Excellent <0.20')
ax2.axhspan(0.20, 0.35, color='gold', alpha=0.12, label='Good 0.20–0.35')
ax2.axhspan(0.35, 0.55, color='orange', alpha=0.12, label='Moderate 0.35–0.55')
ax2.axhspan(0.55, 1.0, color='red', alpha=0.08, label='High >0.55')
if nqe_series:
    ax2.text(0.98, 0.95, f'Final nQE: {nqe_series[-1]:.3f}', transform=ax2.transAxes,
             fontsize=10, fontweight='bold', ha='right', va='top',
             bbox=dict(boxstyle='round,pad=0.35', facecolor='white', alpha=0.85))
ax2.legend(fontsize=8, loc='lower right', frameon=True)

# Plot 3: QE vs KMeans Baseline Ratio
ax3 = fig.add_subplot(gs[0, 2])
if kmeans_baseline and kmeans_baseline > 0 and 'quantization_errors_kmeans_ratio' in globals():
    ax3.plot(x_raw, k_ratio_series, marker='^', color='#2ca02c', linewidth=2.0, markersize=5)
    ax3.axhline(1.0, color='black', linestyle='--', linewidth=1.2, alpha=0.7, label='Parity (SOM = KMeans)')
    ax3.axhline(0.8, color='green', linestyle='--', linewidth=1.0, alpha=0.6, label='Better (<0.8)')
    ax3.axhline(1.2, color='red', linestyle='--', linewidth=1.0, alpha=0.6, label='Worse (>1.2)')
    ax3.set_title('QE Ratio: SOM QE / KMeans QE', fontsize=12, fontweight='bold', pad=16)
    ax3.set_ylabel('QE Ratio', fontsize=11)
    ax3.set_xlabel('Checkpoint', fontsize=11)
    ax3.grid(True, alpha=0.3, linestyle='--')
    if k_ratio_series:
        ax3.text(0.98, 0.95, f'Final Ratio: {k_ratio_series[-1]:.2f}', transform=ax3.transAxes,
                 fontsize=10, fontweight='bold', ha='right', va='top',
                 bbox=dict(boxstyle='round,pad=0.35', facecolor='white', alpha=0.85))
    ax3.legend(fontsize=8, loc='lower right')
else:
    ax3.text(0.5, 0.5, 'KMeans baseline not available', ha='center', va='center', fontsize=12)
    ax3.set_axis_off()

# Plot 4: Topographic Error
ax4 = fig.add_subplot(gs[1, 0])
ax4.plot(x_raw, topographic_errors, marker='d', color='#d62728', linewidth=2.0, markersize=5)
ax4.set_title('Topographic Error During Training', fontsize=12, fontweight='bold', pad=16)
ax4.set_xlabel('Checkpoint', fontsize=11)
ax4.set_ylabel('Topographic Error', fontsize=11)
ax4.grid(True, alpha=0.3, linestyle='--')
ax4.axhline(0.2, color='green', linestyle='--', alpha=0.6, linewidth=1.2, label='Excellent <0.2')
ax4.axhline(0.5, color='orange', linestyle='--', alpha=0.6, linewidth=1.2, label='Acceptable <0.5')
if topographic_errors:
    ax4.text(0.98, 0.95, f'Final TE: {topographic_errors[-1]:.3f}', transform=ax4.transAxes,
             fontsize=10, fontweight='bold', ha='right', va='top',
             bbox=dict(boxstyle='round,pad=0.35', facecolor='white', alpha=0.85))
ax4.legend(fontsize=8, loc='lower right')

# Plot 5: Joint Progress (raw QE + nQE)
ax5 = fig.add_subplot(gs[1, 1])
ln1 = ax5.plot(x_raw, quantization_errors, color='#1f77b4', marker='o', markersize=4, linewidth=1.8, label='Raw QE')
ax5.set_xlabel('Checkpoint', fontsize=11)
ax5.set_ylabel('Raw QE', fontsize=11, color='#1f77b4')
ax5.tick_params(axis='y', labelcolor='#1f77b4', labelsize=9)
ax5.grid(True, alpha=0.3, linestyle='--')
ax5_t = ax5.twinx()
ln2 = ax5_t.plot(x_raw, nqe_series, color='#ff7f0e', marker='s', markersize=4, linewidth=1.8, label='Normalized QE')
ax5_t.set_ylabel('Normalized QE', fontsize=11, color='#ff7f0e')
ax5_t.tick_params(axis='y', labelcolor='#ff7f0e', labelsize=9)
lines = ln1 + ln2
labels = [l.get_label() for l in lines]
ax5.legend(lines, labels, loc='upper right', fontsize=9)
ax5.set_title('Training Progress: Raw vs Normalized QE', fontsize=12, fontweight='bold', pad=16)

# Plot 6: Cluster Load Distribution (reuse existing logic if available)
ax6 = fig.add_subplot(gs[1, 2])
unique_clusters_vals, cluster_counts = np.unique(som_clusters, return_counts=True)
ax6.bar(unique_clusters_vals, cluster_counts, color='#8c6bb1', alpha=0.85, linewidth=0)
ax6.set_title('Cells per SOM Node', fontsize=12, fontweight='bold', pad=16)
ax6.set_xlabel('SOM Node ID', fontsize=11)
ax6.set_ylabel('Cell Count', fontsize=11)
ax6.tick_params(labelsize=8)

max_idx = np.argmax(cluster_counts)
max_pct = cluster_counts[max_idx] / len(original_df) * 100
if max_pct > 25:
    ax6.text(0.98, 0.95, f'Largest node: {max_pct:.1f}%', transform=ax6.transAxes,
             ha='right', va='top', fontsize=9, fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.8), color='red')

fig.suptitle('SOM Training Quality Assessment (Enhanced Metrics)', fontsize=16, fontweight='bold', y=0.97)
plt.show()

# Summary block
print("\n" + "="*90)
print("SOM QUALITY SUMMARY")
print("="*90)
if quantization_errors:
    print(f" Final Raw QE: {quantization_errors[-1]:.3f}")
if nqe_series:
    print(f" Final Normalized QE: {nqe_series[-1]:.3f}")
if kmeans_baseline:
    print(f" KMeans Baseline QE (eval): {kmeans_baseline:.3f}")
if k_ratio_series:
    print(f" Final QE/KMeans Ratio: {k_ratio_series[-1]:.2f}")
if topographic_errors:
    print(f" Final TE: {topographic_errors[-1]:.3f}")

# Interpretations
if nqe_series:
    nqe_final = nqe_series[-1]
    if nqe_final < 0.20:
        print(" Quantization Quality: Excellent")
    elif nqe_final < 0.35:
        print(" Quantization Quality: Good")
    elif nqe_final < 0.55:
        print(" Quantization Quality: Moderate")
    else:
        print(" Quantization Quality: Needs improvement")

if topographic_errors:
    te_final = topographic_errors[-1]
    if te_final < 0.2:
        print(" Topology Preservation: Excellent")
    elif te_final < 0.5:
        print(" Topology Preservation: Acceptable")
    else:
        print(" Topology Preservation: Weak (consider larger sigma early)")

if k_ratio_series:
    ratio_final = k_ratio_series[-1]
    if ratio_final < 0.8:
        print(" SOM Outperforms equivalent k-means (ratio < 0.8)")
    elif ratio_final < 1.1:
        print(" SOM Comparable to k-means (ratio < 1.1)")
    else:
        print(" SOM Underperforming vs k-means (ratio >= 1.1) → consider more iterations or finer sigma schedule")

print("="*90)


## Step 4: Expression Heatmap Generation

Generate a clustered heatmap to visualize meta-cluster marker expressions for phenotype assignment.

In [ ]:
# =============================================================================
# STEP 4: EXPRESSION HEATMAP FOR PHENOTYPE ASSIGNMENT
# =============================================================================
print("Step 4: Creating expression heatmap for phenotype assignment...")

def create_clustered_heatmap(cluster_data_df, title="Meta-Cluster Expression Heatmap", 
                             save_name="meta_cluster_heatmap.png"):
    """Generate clustered heatmap for meta-cluster marker expression."""
    # Extract mean expression columns
    marker_cols = [c for c in cluster_data_df.columns if c.endswith('_mean')]
    heatmap_data = cluster_data_df.set_index('som_cluster')[marker_cols].copy()
    
    # Clean marker names
    clean_names = {col: col.replace('_corr_norm_mean','')
                      .replace('_corr_mean','')
                      .replace('_mean','')
                  for col in marker_cols}
    heatmap_data.rename(columns=clean_names, inplace=True)
    
    # Transpose: clusters on X-axis, markers on Y-axis
    heatmap_data = heatmap_data.T
    
    # Z-score normalization by marker (row-wise)
    heatmap_z = heatmap_data.apply(lambda x: (x - x.mean())/x.std(), axis=1).fillna(0)
    
    # Calculate appropriate figure size
    n_clusters = heatmap_z.shape[1]
    n_markers = heatmap_z.shape[0]
    cell_size = 0.3
    fig_w = max(8, n_clusters * cell_size)
    fig_h = max(6, n_markers * cell_size)
    
    # Create clustered heatmap
    g = sns.clustermap(
        heatmap_z,
        method='ward',
        metric='euclidean',
        cmap='bwr',
        vmin=-4, vmax=4,
        linewidths=0.5,
        row_cluster=True,
        col_cluster=True,
        dendrogram_ratio=(0.05, 0.2),
        cbar_pos=(0.00, 0.00, 0.01, 0.10),
        cbar_kws={'label':'Z-score','shrink':0.5},
        figsize=(fig_w, fig_h),
    )
    
    # Add title and labels
    g.fig.suptitle(f"{title} (n={len(cluster_data_df)})", y=1.02, fontsize=16)
    g.ax_heatmap.set_xlabel('Meta-Clusters')
    g.ax_heatmap.set_ylabel('Markers')
    
    # Move marker labels to right side
    g.ax_heatmap.yaxis.tick_right()
    g.ax_heatmap.yaxis.set_label_position('right')
    
    # Add cell counts to cluster labels
    sizes = cluster_data_df.set_index('som_cluster')['cell_count']
    order = [int(t.get_text()) for t in g.ax_heatmap.get_xticklabels()]
    new_labels = [f"Cluster {i} ({sizes[i]:,} cells)" for i in order]
    g.ax_heatmap.set_xticklabels(new_labels, rotation=90, ha='right')
    
    # Save heatmap
    g.fig.tight_layout()
    output_path = PLOTS_DIR / save_name
    g.fig.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"Heatmap saved: {output_path}")
    plt.show()
    
    return order

# Generate the clustered heatmap
print(f"Generating heatmap for {len(char_df)} meta-clusters...")
reordered_cluster_ids = create_clustered_heatmap(
    char_df, 
    title=f"Meta-Cluster Expression Heatmap (n={len(char_df)})",
    save_name="meta_cluster_expression_heatmap.png"
)

## Step 5: Interactive Phenotype Assignment

**Professional phenotype assignment interface**

This step provides an interactive GUI for efficient phenotype assignment using the expression heatmap from Step 4. 

**Features:**
- **Interactive heatmap**: Click cluster columns for selection (yellow highlighting)
- **Real-time updates**: Phenotype heatmap updates with assignments
- **Table interface**: View and manage assignments in organized table
- **Professional workflow**: Consistent with Leiden method interface

**Workflow Options:**
- **Option A**: Use this interactive GUI → Skip Step 6 → Proceed to Step 7
- **Option B**: Skip this step → Use Step 6 for manual assignment → Proceed to Step 7

The GUI uses the exact heatmap ordering from Step 4 for consistency.

In [ ]:
# =============================================================================
# STEP 5: INTERACTIVE PHENOTYPE ASSIGNMENT
# =============================================================================
print("Step 5: Launching interactive phenotype assignment studio...")

# Initialize phenotype assignments
if 'META_CLUSTER_PHENOTYPES' not in globals():
    META_CLUSTER_PHENOTYPES = {}

# Launch professional assignment interface
META_CLUSTER_PHENOTYPES = launch_som_phenotype_assignment_app(
    char_df, reordered_cluster_ids, META_CLUSTER_PHENOTYPES
)

print(f"Interactive assignment complete: {len(META_CLUSTER_PHENOTYPES)} meta-clusters assigned")

## Step 6: Manual Phenotype Assignment (Optional)

Alternative manual phenotype assignment for users who prefer scripted assignments.

In [ ]:
# =============================================================================
# STEP 6: MANUAL PHENOTYPE ASSIGNMENT (OPTIONAL)
# =============================================================================
print("Step 6: Manual phenotype assignment (optional)")

# Example manual assignment approach:
# Uncomment and edit the following lines to make manual assignments

# if 'META_CLUSTER_PHENOTYPES' not in globals():
#     META_CLUSTER_PHENOTYPES = {}

# example_assignments = {
#     'Granulocytes': [64, 35, 2, 45, 24, 8, 71, 60, 77, 56, 58],
#     'Cytotoxic T': [85, 54, 63, 34],
#     'B Cell': [28, 55, 91],
#     'Epithelia': [1, 5, 12, 20],
# }

# for phenotype, clusters in example_assignments.items():
#     for cluster_id in clusters:
#         META_CLUSTER_PHENOTYPES[cluster_id] = phenotype

# print(f"Manual assignments complete: {len(META_CLUSTER_PHENOTYPES)} clusters assigned")

# Coverage check
# all_clusters = set(char_df['som_cluster'])
# assigned_clusters = set(META_CLUSTER_PHENOTYPES.keys())
# unassigned = all_clusters - assigned_clusters
# print(f"Unassigned clusters: {len(unassigned)}")

print("No manual assignments made. Use Step 5 interactive GUI or uncomment code above.")

## Step 7: Apply Phenotypes and Save Results

Apply the phenotype assignments to the original dataset and save final results with proper data preservation.

In [ ]:
# =============================================================================
# STEP 7: APPLY PHENOTYPES & SAVE UNIFIED OUTPUTS
# =============================================================================
print("Step 7: Applying phenotype assignments & saving results...")

if not META_CLUSTER_PHENOTYPES:
    print("Warning: No phenotype assignments found. Please complete Step 5 or Step 6 first.")
    print("Available assignment steps:")
    print(" - Step 5: Interactive GUI assignment")
    print(" - Step 6: Manual scripted assignment")
else:
    print(f"Applying {len(META_CLUSTER_PHENOTYPES)} phenotype assignments...")
    
    # Apply phenotypes using clean data without normalized columns
    # FIXED: Filter out normalized columns to prevent them from being saved to unified CSV
    clean_data = original_df[[c for c in original_df.columns if not c.endswith('_norm')]].copy()
    
    result = apply_som_phenotypes(
        data=clean_data,
        som_clusters=som_clusters,
        meta_cluster_map=meta_cluster_map,
        meta_cluster_phenotypes=META_CLUSTER_PHENOTYPES,
        output_base_dir=OUTPUT_DIR,  # FIXED: Use OUTPUT_DIR instead of default_config.output_dir
        cell_id_column=CELL_ID_COLUMN,  # UPDATED: Required parameter for configurable cell ID column
        method_prefix='som'
    )

    print(f"\nStep 7 Results:")
    print(f" Unified file: {result['unified_file']}")
    print(f" Summary file: {result['summary_file']}")
    print(f" Mapping file: {result['mapping_file']}")
    print(f" SOM meta-cluster map: {result['som_meta_file']}")
    
    print(f"\nPhenotype Summary:")
    summary_display = result['summary_df'].head(10)
    for _, row in summary_display.iterrows():
        phenotype = row.name
        count = int(row['cell_count'])
        pct = row['percentage']
        print(f" {phenotype}: {count:,} cells ({pct:.1f}%)")
    
    if len(result['summary_df']) > 10:
        print(f" ... and {len(result['summary_df']) - 10} more phenotypes")
    
    # Validation: Check unified file integrity
    unified_df = result['data']
    print(f"\nData Integrity Check:")
    print(f" Original cells: {len(original_df):,}")
    print(f" Unified output cells: {len(unified_df):,}")
    print(f" Cell ID preservation: {unified_df[CELL_ID_COLUMN].nunique():,} unique IDs")
    
    if 'phenotype_som' in unified_df.columns:
        print(f" SOM phenotype column added successfully")
        unassigned = unified_df['phenotype_som'].isnull().sum()
        if unassigned > 0:
            print(f" Warning: {unassigned} cells have no phenotype assignment")
        else:
            print(f" All cells successfully assigned phenotypes")
    else:
        print(" Warning: phenotype_som column not found in output")
        
    print(f"\nSOM phenotyping complete. Results saved to: {result['unified_file']}")

## Step 8: Optional Batch Export

Export individual batch files for downstream analysis or validation.

**Export Configuration:**
- Set `EXPORT_ALL_BATCHES = True` to export all batches
- Set `EXPORT_ALL_BATCHES = False` and specify `EXPORT_PREFIXES` for selective export
- Files are saved to `phenotyping/som/batch/` directory

This step is optional and doesn't affect the main unified output.

In [ ]:
# =============================================================================
# STEP 8: OPTIONAL BATCH EXPORT
# =============================================================================
import re
from pathlib import Path

# Export configuration
EXPORT_ALL_BATCHES = False       # Set True to export every batch
EXPORT_PREFIXES = ['P11_reoi_1']         # Prefixes to match when EXPORT_ALL_BATCHES is False

# Setup paths
BATCH_EXPORT_DIR = default_config.output_dir / 'phenotyping/som/batch'
BATCH_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

UNIFIED_FILE = default_config.output_dir / 'phenotyping/all_samples_combined_corrected_phenotyped.csv'
BATCH_COL = 'batch_id'
PHENOTYPE_COL = 'phenotype_som'

print("Step 8: Optional batch export...")

# Load unified file
if 'result' in globals() and isinstance(result, dict) and 'unified_file' in result:
    unified_path = Path(result['unified_file'])
else:
    unified_path = UNIFIED_FILE

if not unified_path.exists():
    print(f"Unified file not found: {unified_path}")
    print("Run Step 7 first to generate unified output file.")
else:
    unified_df = pd.read_csv(unified_path)
    print(f"Loaded unified file: {unified_path} ({len(unified_df):,} rows)")

    if BATCH_COL not in unified_df.columns:
        print(f"Column '{BATCH_COL}' not found; skipping batch export.")
    else:
        # Determine target batches
        batches = unified_df[BATCH_COL].dropna().astype(str).unique().tolist()
        
        if EXPORT_ALL_BATCHES:
            target_batches = batches
            print(f"Exporting ALL {len(target_batches)} batches...")
        else:
            target_batches = []
            for prefix in EXPORT_PREFIXES:
                matched = [b for b in batches if b.startswith(prefix)]
                target_batches.extend(matched)
            
            # Remove duplicates while preserving order
            seen = set()
            target_batches = [b for b in target_batches if not (b in seen or seen.add(b))]
            print(f"Exporting {len(target_batches)} batches matching prefixes {EXPORT_PREFIXES}")

        if not target_batches:
            print('No batches match the export criteria.')
        else:
            # Export each batch
            for batch_id in target_batches:
                batch_df = unified_df[unified_df[BATCH_COL].astype(str) == batch_id].copy()
                
                # Create safe filename
                safe_name = re.sub(r'[\\/*?:"<>|\s]+', '_', batch_id)
                output_file = BATCH_EXPORT_DIR / f'{safe_name}_phenotyped_som.csv'
                
                batch_df.to_csv(output_file, index=False)
                print(f" Exported {batch_id}: {len(batch_df):,} cells → {output_file.name}")
            
            print(f"\nBatch export complete. Files saved to: {BATCH_EXPORT_DIR}")

print("Step 8 complete.")

## Step 9: Cleanup Normalized Cache

Delete the normalized cache file to ensure fresh computation on next run. This prevents cache inconsistencies and ensures reproducibility.

In [ ]:
# =============================================================================
# STEP 9: CLEANUP NORMALIZED CACHE
# =============================================================================
print("Step 9: Cleaning up normalized cache...")

# Delete the shared normalization cache after processing is complete
if 'cache_csv' in globals() and cache_csv.exists():
    cache_csv.unlink()
    print(f"Deleted normalized cache: {cache_csv}")
    
    # Also clean up the cache directory if it's empty
    cache_dir = cache_csv.parent
    if cache_dir.exists() and not any(cache_dir.iterdir()):
        cache_dir.rmdir()
        print(f"Removed empty cache directory: {cache_dir}")
else:
    print("No cache file found to clean up")
    
print("Cache cleanup complete. Fresh normalization will be computed on next run.")

## Summary

This notebook performs SOM-based phenotyping with a professional, publication-ready structure identical to the Leiden clustering method.

## Major Improvements Implemented

### Large Cluster Formation Issue Resolution
**Problem Identified**: Previous configuration led to over-aggregation with one meta-cluster containing approximately 30% of cells due to:
- Small SOM grid (30×30 = 900 nodes) 
- Too few node clusters (80 for 900 nodes)
- Aggressive Ward clustering for meta-clustering

**Solutions Implemented**:
- **Increased SOM grid**: 30×30 to 50×50 (2500 nodes for better granularity)
- **More node clusters**: 80 to 200 (better intermediate resolution)
- **Increased meta-clusters**: 80 to 150 (preserves biological diversity)
- **Gentler clustering**: Ward to Average linkage with correlation distance
- **Better topology**: Increased sigma (1.8 to 2.5) for improved neighborhood preservation

### Training Metrics Understanding Enhancement
**Added comprehensive explanations**:
- **Quantization Error**: Measures data representation quality (lower values preferred)
- **Topographic Error**: Measures topology preservation (lower values preferred)
- **Clinical interpretation**: How metrics relate to biological clustering quality
- **Publication-ready visualizations**: Professional plots with annotations and interpretations

### Normalization Consistency Verification
**Confirmed identical normalization with Leiden method**:
- Shared normalization cache prevents discrepancies
- Avoided double-normalization through proper scaling detection
- Cell_ID anchoring ensures perfect alignment across methods

### Cell_ID Anchoring Implementation
**Problem Identified**: Previous versions used positional cluster assignment, which could cause cell-cluster misalignment when `pd.merge()` operations reordered DataFrame rows.

**Solution Implemented**: 
- Modified `train_som()` function to return cluster assignments with cell_id anchoring
- Updated notebook to use cell_id-based merging instead of positional assignment
- Added comprehensive validation to verify correct cell-cluster mapping

**Result**: 100% accurate cluster assignments anchored to cell_id for data integrity

### Workflow Steps
1. **Data Preparation** - Load batch-corrected data with flexible filtering
2. **SOM Training** - Geometric sketching, self-organizing map with node clustering (Cell_ID anchoring implemented)
3. **Cluster Characterization** - Expression profiling and hierarchical meta-clustering  
4. **Expression Heatmap** - Clustered visualization for phenotype assignment
5. **Interactive Assignment** - Professional GUI for efficient phenotype annotation
6. **Manual Assignment** - Optional scripted assignment alternative
7. **Apply Phenotypes** - Generate unified output with preserved data integrity
8. **Batch Export** - Optional individual batch file generation
9. **Cache Cleanup** - Remove normalized cache for fresh computation on next run

### Key Outputs
**Main Results** (stored in `phenotyping/`)
- `all_samples_combined_corrected_phenotyped.csv` - Unified dataset with SOM phenotypes
- `som_phenotype_summary.csv` - Phenotype statistics and cell counts
- `som_phenotype_mapping.csv` - Meta-cluster to phenotype mapping
- `som_to_meta_cluster_mapping.csv` - SOM cluster hierarchy

**Visualizations** (stored in `phenotyping/som/plots/`)
- Meta-cluster expression heatmaps
- Publication-ready SOM training quality assessment plots
- Interactive phenotype assignment interface

### Key Features
- **Large cluster prevention**: Improved parameters prevent over-aggregation
- **Cell_ID anchoring**: Guaranteed correct cluster-cell alignment using cell_id-based merging
- **Publication-ready metrics**: Professional training quality assessment with interpretations
- **Shared normalization cache**: Consistent with Leiden method for reproducibility
- **Geometric sketching**: Representative subsampling preserves biological diversity
- **Professional interface**: Interactive GUI matching Leiden workflow
- **Data integrity**: Preserves spatial coordinates and cell ID anchoring
- **Flexible configuration**: Configurable filtering, clustering parameters, and export options
- **Cache management**: Automatic cleanup ensures fresh computation on next run

### Expected Results After Improvements
- **Balanced cluster sizes**: No single cluster should contain more than 20% of cells
- **Better topology preservation**: Lower topographic error with increased grid size
- **Improved biological relevance**: More diverse intermediate clusters before meta-clustering
- **Publication quality**: Professional visualizations with clear metric interpretations

### Next Steps
The unified output file is ready for downstream spatial analysis and can be combined with results from other phenotyping methods (Leiden, threshold-based) for comprehensive validation and analysis. 

**Recommended workflow**:
1. Run the improved notebook with new parameters
2. Verify cluster size distribution in Step 3.2 assessment
3. Proceed with phenotype assignment using balanced clusters
4. Compare results with Leiden method for validation